# Data Cleaning & Tidy Process

### Load the data
Here we would like to load our olympics_08_medalists dataset and import the pandas library necessary for data cleaning and manipulation.

In [ ]:
import pandas as pd
# Load the data
data = pd.read_csv('olympics_08_medalists.csv')

### Reshaping the Data
This is an important step for data analysis and our tidy data process.

Working in accordance with our tidy data principles, we want to ensure that each variable has its own column, each observation forms its own row, and each type of observational unit forms its own table.

Looking through the data that we have just loaded, we can see that the columns are an issue. They have both the gender and the sport of the athlete in the same cell. Let's identify those columns first so we can separate them later on.

In [ ]:
# Create a list of the columns that contain the gender and sport information, which are all columns except for 'medalist_name'.
gendersport_columns = [col for col in data.columns if col != 'medalist_name']

Now, we want to melt the data so that we have one row per medalist. We want to convert the gendersport_columns into one column called gender_sport and add a new column for our medal values.

In [ ]:
# Use the melt function to reshape the data from wide to long format. The id_vars specifies the column that will remain fixed (medalist_name), the value_vars specifies the columns that will be stacked into rows (gendersport_columns), var_name specifies the name of the new column that will contain the old column names, and value_name will create a column storing the values from the old columns.
data_long = data.melt(id_vars = ['medalist_name'], value_vars = gendersport_columns, var_name = "gender_sport", value_name = "medal")

There are a lot of NaN values in the medal column, so let's drop those rows.

In [ ]:
# Use the dropna function to remove rows where the medal column is NaN
data_long = data_long.dropna(subset=['medal'])

Now, we still want to actually split the gender and sport into separate columns by using str.split()

In [ ]:
data_long[['gender', 'sport']] = data_long['gender_sport'].str.split('_', expand=True)

Since we have separated gender and sport into separate columns, we can drop the initial column we created where they were combined.

In [ ]:
data_long = data_long.drop(columns=['gender_sport'])

Finally, let's look at our transformed data to make sure it looks correct. Again, we want each variable in its own column, each observation in its own row, and each type of observational unit to form its own table.

In [17]:
print(data_long.head())

      medalist_name   medal gender    sport
177    Bair Badënov  bronze   male  archery
676   Ilario Di Buò  silver   male  archery
682    Im Dong-hyun    gold   male  archery
760       Jiang Lin  bronze   male  archery
920  Lee Chang-hwan    gold   male  archery


### Data Visualization
Now that we have the data in a tidy format, we can analyze and visualize it.

Start by importing common Python libraries used for visualization

In [ ]:
# Import seaborn and matplotlib for visualization
import matplotlib.pyplot as plt
import seaborn as sns

Next, we'll take a look at how many medals were won in each sport

In [ ]:
# Use seaborn to create a countplot of the number of medals won by sport.
plt.figure(figsize=(10,6))
sns.countplot(data=data_long, x='sport', order=data_long['sport'].value_counts().index)
plt.xticks(rotation=90)
plt.title('Number of Medals Won by Sport')
plt.xlabel('Sport')
plt.ylabel('Number of Medals')
plt.show()

It looks like swimming, rowing, and athletics had the most medals won in the '08 Olympics.

What about the number of medals won by gender?

In [ ]:
# Use seaborn to create a countplot of the number of medals
plt.figure(figsize=(6,4))
sns.countplot(data=data_long, x='gender')
plt.title('Number of Medals Won by Gender')
plt.xlabel('Gender')
plt.ylabel('Number of Medals')
plt.show()

It looks like there were more medals won by males in the '08 Olympics, but there were still a lot of medals won by females.
But what if we break this down a little further? Let's look at how many gold medals were won by each gender.

In [ ]:
# Use seaborn to create a countplot of the number of gold medals
plt.figure(figsize=(6,4))
sns.countplot(data=data_long[data_long['medal'] == 'gold'], x='gender')
plt.title('Number of Gold Medals Won by Gender')
plt.xlabel('Gender')
plt.ylabel('Number of Gold Medals')
plt.show()

In [ ]:
# Percentage breakdown of those numbers
gold_medals = data_long[data_long['medal'] == 'gold']
gold_medal_counts = gold_medals['gender'].value_counts()
gold_medal_percentages = gold_medal_counts / gold_medals.shape[0] * 100
print(gold_medal_percentages)

### Pivot Tables
Lastly, we can summarize our data in an easy-to-read format using pivot tables. This can help us discover important trends in the data very quickly.

In [ ]:
# Let's look at medals by gender and sport
medal_pivot = data_long.pivot_table(index='sport', columns='gender', values='medal', aggfunc='count', fill_value=0)
print(medal_pivot)  

Here we can more easily view medal winnings by sport and gender. 

In [ ]:
# We can also look at total medals by medal type
medal_type_pivot = data_long.pivot_table(index='medal', values='medalist_name', aggfunc='count')
print(medal_type_pivot)

As we might suspect, there are more bronze medals won than silver medals, and more silver medals won than gold medals.
